# **Running Difference**

**Level 0.5**  

In [ ]:
'''
Written by: Pritam Das, ARIES, Nainital, India
Date: 2026-03-31
Version: DOFCAT Preprocessing 2.0 [LASCO C2]

Description: This script processes SOHO/LASCO C2 FITS files to create running difference images. It performs the following steps:
1. Loads all FITS files from a specified folder and filters 1024x1024 frames.
2. Removes cosmic rays (via Astro-SCRAPPY) and normalizes/subtracts local background (via Gaussian blur).
3. Computes running difference images between consecutive frames.
4. Corrects roll angle by rotating difference maps with SunPy and masks out the central occulter.
5. Converts pixel coordinates into physical coordinates in solar radii (R_sun) for accurate plotting.
6. Draws a circle representing the solar disk (1 Rsun) on the images.
7. Plots and saves all running difference images with consistent display scaling.
8. Saves the difference images and headers in a compressed format for future use.
'''

import os
import pickle
import numpy as np
import cv2
from pathlib import Path
from astropy.io import fits
from astropy.time import Time
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
import sunpy.map
import astroscrappy


# Load and preprocess all LASCO C2 FITS files from a folder
def open_lasco_fits_files(path):
    lasco_all = []
    header_all = []

    # Collect all FITS files
    files = sorted(Path(path).glob("*"))
    valid_files = [f for f in files if f.suffix.lower() in ['.fts', '.fits']]

    if len(valid_files) == 0:
        raise FileNotFoundError(f"No .fts or .fits files found in {path}")

    # Read image data, reject cosmics, and subtract background
    for location in valid_files:
        print(f"Processing FITS file: {location.name}")
        with fits.open(location) as hdul:
            for hdu in hdul:
                if hdu.data is not None and hdu.data.shape == (1024, 1024):
                    # Step 1: Remove cosmic rays
                    mask, clean_data = astroscrappy.detect_cosmics(
                        hdu.data,
                        sigclip=2,
                        objlim=2,
                        readnoise=4,
                        verbose=False
                    )

                    # Step 2: Normalize and subtract background
                    mean_value = np.mean(clean_data)
                    img_normalized = clean_data / mean_value
                    background = cv2.GaussianBlur(img_normalized, (0, 0), 100)
                    subtracted_img = img_normalized - background

                    lasco_all.append(subtracted_img)
                    header_all.append(hdu.header)

    return lasco_all, header_all


# Build a circular mask to block out the inner occulter
def create_circular_mask(shape, center, radius=180):
    Y, X = np.ogrid[:shape[0], :shape[1]]
    dist_from_center = np.sqrt((X - center[0])**2 + (Y - center[1])**2)
    mask = dist_from_center <= radius
    return mask


# Convert pixel grid into physical coordinates from rotated header
def get_extent_info(header, shape):
    crval1 = float(header.get('CRVAL1', 0.0))
    crval2 = float(header.get('CRVAL2', 0.0))
    cdelt1 = float(header.get('CDELT1', 1.0))
    cdelt2 = float(header.get('CDELT2', 1.0))
    crpix1 = float(header.get('CRPIX1', shape[1] / 2.0))
    crpix2 = float(header.get('CRPIX2', shape[0] / 2.0))

    n_rows, n_cols = shape

    # In FITS convention (1-based), pixel (0,0) center is at (1, 1),
    # so lower-left outer corner is (0.5, 0.5), upper-right outer corner is (n_cols + 0.5, n_rows + 0.5)
    x_min = crval1 + (0.5 - crpix1) * cdelt1
    x_max = crval1 + (n_cols + 0.5 - crpix1) * cdelt1
    y_min = crval2 + (0.5 - crpix2) * cdelt2
    y_max = crval2 + (n_rows + 0.5 - crpix2) * cdelt2

    orgn = [x_min, y_min]
    ep = [x_max, y_max]
    aspect_ratio = (x_max - x_min) / (y_max - y_min)

    return orgn, ep, aspect_ratio


# Draw a circle representing the solar disk (1 Rsun)
def add_solar_disk_circle(ax, header):
    crval1 = float(header.get('CRVAL1', 0.0))
    crval2 = float(header.get('CRVAL2', 0.0))
    cdelt1 = float(header.get('CDELT1', 1.0))
    cdelt2 = float(header.get('CDELT2', 1.0))
    crpix1 = float(header.get('CRPIX1', 0.0))
    crpix2 = float(header.get('CRPIX2', 0.0))

    sun_xcen = float(header.get('SUNPIX1', crpix1))
    sun_ycen = float(header.get('SUNPIX2', crpix2))

    rsun_arcsec = float(header.get('RSUN', header.get('RSUN_ARC', header.get('RSUN_OBS', 959.63))))

    # Convert Sun center into world coordinates
    x_world = crval1 + (sun_xcen - crpix1) * cdelt1
    y_world = crval2 + (sun_ycen - crpix2) * cdelt2

    # Express in solar radii
    x_rs = x_world / rsun_arcsec
    y_rs = y_world / rsun_arcsec

    circle = Circle(
        (x_rs, y_rs),
        radius=1.0,
        transform=ax.transData,
        edgecolor='white',
        facecolor='none',
        linewidth=1.5,
        linestyle='--'
    )

    ax.add_patch(circle)


# Plot and save all running difference images
def plot_all_lasco_diff_images(diff_imgs, header_all, save_dir):

    os.makedirs(save_dir, exist_ok=True)
    os.makedirs(os.path.join(save_dir, "difference_images"), exist_ok=True)

    # Use global scaling across frames
    vmin = np.min([np.min(diff_img) * np.exp(-9.0) for diff_img in diff_imgs])
    vmax = np.max([np.max(diff_img) * np.exp(-4.0) for diff_img in diff_imgs])

    print(f"\nFinal display limits:")
    print(f"vmin = {vmin:.6e}")
    print(f"vmax = {vmax:.6e}")

    dpi = 100
    image_area_pixels = 1024
    label_padding = 200
    total_size = image_area_pixels + label_padding
    figsize = (total_size / dpi, total_size / dpi)

    for i, (diff_img, header) in enumerate(zip(diff_imgs, header_all)):

        # Compute exact extent for this rotated frame
        orgn, ep, aspect_ratio = get_extent_info(header, diff_img.shape)

        # Read observation time if available
        try:
            date_part = str(header.get('DATE-OBS', header.get('DATE', ''))).strip()
            time_part = str(header.get('TIME-OBS', '')).strip()

            if '/' in date_part:
                date_part = date_part.replace('/', '-')

            if time_part:
                date_obs_full = f"{date_part}T{time_part}"
            else:
                date_obs_full = f"{date_part}T00:00:00"

            t = Time(date_obs_full, format='isot', scale='utc')
            time_str = t.strftime('%Y-%m-%d %H:%M:%S UT')
        except Exception as e:
            print(f"Warning: Could not parse date from header, error: {e}")
            time_str = "Unknown Time"

        fig = plt.figure(figsize=figsize, dpi=dpi)

        left = label_padding / 2 / total_size
        bottom = label_padding / 2 / total_size
        size = image_area_pixels / total_size
        ax = fig.add_axes([left, bottom, size, size])

        # Convert extent into solar radii (R_sun)
        rsun_arcsec = float(header.get('RSUN', header.get('RSUN_ARC', header.get('RSUN_OBS', 959.63))))
        extent_rs = [val / rsun_arcsec for val in [orgn[0], ep[0], orgn[1], ep[1]]]

        ax.imshow(
            diff_img,
            cmap='gray',
            vmin = -1.516202e-04,
            vmax = 2.238813e-02,
            origin='lower',
            extent=extent_rs,
            aspect=aspect_ratio
        )

        # Overlay solar disk (1 Rsun)
        add_solar_disk_circle(ax, header)

        ax.set_title(f'LASCO C2 Frame {i:04d}\n{time_str}', fontsize=16, pad=20)
        ax.set_xlabel(r'Solar X [R$_\odot$]', fontsize=14)
        ax.set_ylabel(r'Solar Y [R$_\odot$]', fontsize=14)
        ax.tick_params(labelsize=12)

        plt.savefig(
            f"{save_dir}/difference_images/difference_image_{i:04d}.png",
            dpi=dpi,
            pad_inches=0
        )
        plt.close()

    # Save headers for later use
    with open(f"{save_dir}/difference_images/difference_headers.pkl", "wb") as f:
        pickle.dump(header_all, f)


# Main pipeline
def create_lasco_running_difference(path, save_dir):

    lasco_all, header_all = open_lasco_fits_files(path)

    # Check exposure times
    exposure_times = [header.get('EXPTIME') for header in header_all]

    if len(set(exposure_times)) > 1:
        print("Warning: Exposure times vary between frames:")
        for i, exptime in enumerate(exposure_times):
            print(f"Frame {i:04d}: EXPTIME = {exptime}")
    else:
        print(f"All frames have same exposure time: {exposure_times[0] if exposure_times else 'N/A'}")

    diff_imgs = []
    valid_headers = []

    # Running difference using consecutive frames i and i+1
    for i in range(len(lasco_all) - 1):
        img1 = lasco_all[i]
        img2 = lasco_all[i + 1]

        if img1.shape != img2.shape:
            print(f"Skipping pair {i}, shape mismatch: {img1.shape} vs {img2.shape}")
            continue

        difference_image = img2 - img1

        # Create SunPy Map from difference image + header and rotate
        diff_map = sunpy.map.Map(difference_image.astype(np.float32), header_all[i + 1])
        diff_rotated = diff_map.rotate(recenter=True, missing=0)
        difference_image = diff_rotated.data

        # Use rotated header containing the derotated WCS coordinates
        header_rot = diff_rotated.meta

        # Mask out central occulter at the Sun center in the rotated frame
        center_x = header_rot['CRPIX1'] - 1  # 0-based index
        center_y = header_rot['CRPIX2'] - 1  # 0-based index
        radius = 180
        mask = create_circular_mask(difference_image.shape, (center_x, center_y), radius)
        difference_image[mask] = 0

        diff_imgs.append(difference_image)
        valid_headers.append(header_rot)

    if not diff_imgs:
        print("No valid difference images generated.")
        return [], [], [], None, None

    orgn, ep, aspect_ratio = get_extent_info(valid_headers[0], diff_imgs[0].shape)

    plot_all_lasco_diff_images(
        diff_imgs,
        valid_headers,
        save_dir
    )

    return diff_imgs, lasco_all, valid_headers, orgn, ep


if __name__ == "__main__":

    # Folder containing input FITS files
    path = "/full/path/to/input/data/"

    # Folder where results will be saved
    save_dir = "/full/path/to/output/results/"

    diff_imgs, lasco_all, header_all, orgn, ep = create_lasco_running_difference(path, save_dir)

    # Save all difference images in compressed format
    np.savez_compressed(
        os.path.join(save_dir, "difference_images/diff_imgs.npz"),
        **{f"diff_{i:04d}": img.astype(np.float32)
           for i, img in enumerate(diff_imgs)}
    )

# **Optical Flow**

In [ ]:
'''
Written by: Pritam Das, ARIES, Nainital, India
Date: 2026-03-31
Version: DOFCAT Optical Flow 1.0 [LASCO C2]

Description: This script takes the running difference images generated from preprocessing of SOHO/LASCO C2 data and
applies dense optical flow techniques to compute the velocity field of the observed CME. The main steps include:
1. Loading the preprocessed difference images.
2. Applying Bilateral filter and median blur to reduce noise while preserving CME features.
3. Extracting a region of interest (ROI). Change it as per the requirement. Larger ROIs are more computationally expensive, but captures more motion.
4. Computing dense optical flow using Farneback's method to get pixel displacements.
5. Converting pixel displacements into physical velocities (km/s) using the FITS header information.
6. Applying noise filtering and pixel displacement thresholding to retain only physically meaningful velocities.
7. Plotting the velocity magnitude as a heatmap with overlaid quiver vectors in solar radii (R_sun).
8. Overlaying velocity vectors onto the original difference frames.
9. Saving the resulting visualizations and numerical data for further analysis.
'''

import os
import pickle
from glob import glob
import cv2
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from mpl_toolkits.axes_grid1 import make_axes_locatable
from astropy.time import Time


# ------------------------------------------------------------
# Load PNG frames (difference images)
# ------------------------------------------------------------
def load_frames_from_folder(folder_path):

    # Collect all PNG files and sort them chronologically
    file_paths = sorted(glob(os.path.join(folder_path, '*.png')))

    frames = []
    for file_path in file_paths:
        frame = cv2.imread(file_path, cv2.IMREAD_GRAYSCALE)

        # Skip corrupted/unreadable files
        if frame is not None:
            frames.append(frame)

    return frames


# ------------------------------------------------------------
# Basic denoising to reduce small-scale noise
# ------------------------------------------------------------
def reduce_noise(img):

    # Bilateral filter preserves edges while smoothing
    denoised = cv2.bilateralFilter(img, 35, 60, 60)
    # Median blur suppresses residual shot noise
    denoised = cv2.medianBlur(denoised, 5)
    return denoised


# ------------------------------------------------------------
# Extract Region of Interest (ROI)
# ------------------------------------------------------------
def set_ROI(frame, x_value, y_value, width, height):

    # Simple rectangular crop
    return frame[y_value:y_value + height, x_value:x_value + width]


# ------------------------------------------------------------
# Extract observation time from FITS header
# ------------------------------------------------------------
def extract_datetime_from_header(header):

    try:
        date_part = str(header.get('DATE-OBS', header.get('DATE', ''))).strip()
        time_part = str(header.get('TIME-OBS', '')).strip()

        if '/' in date_part:
            date_part = date_part.replace('/', '-')

        if time_part:
            date_obs_full = f"{date_part}T{time_part}"
        else:
            date_obs_full = f"{date_part}T00:00:00"

        date_time = Time(date_obs_full, format='isot', scale='utc')
        return date_time.strftime('%Y-%m-%d %H:%M:%S UT')
    except Exception as e:
        print(f"Warning: Could not parse date/time: {e}")
        return "Unknown Time"


# ------------------------------------------------------------
# Optical flow computation (core algorithm untouched)
# ------------------------------------------------------------
def compute_optical_flow_and_magnitude(frames, headers,
                                       x_value, y_value, width, height,
                                       lower_velocity=50, upper_velocity=700):

    magnitudes = []
    u_list = []
    v_list = []

    # First frame ROI
    prev_frame = set_ROI(frames[0], x_value, y_value, width, height)

    for i in range(1, len(frames)):

        # Next frame ROI
        next_frame = set_ROI(frames[i], x_value, y_value, width, height)

        # Dense optical flow (Farneback)
        flow = cv2.calcOpticalFlowFarneback(prev_frame, next_frame, None,
                                            0.3, 4, 15, 4, 5, 1.1, 0)

        u = flow[..., 0]
        v = flow[..., 1]

        # Pixel displacement magnitude
        velocity = np.sqrt(u**2 + v**2)

        # --------------------------------------------------------
        # Convert pixel displacement -> physical velocity (km/s)
        # --------------------------------------------------------
        cdelt = float(headers[0].get('CDELT1', headers[0].get('cdelt1', 11.9)))
        # Physical size per pixel at 1 AU (1 arcsec ≈ 725 km)
        D = cdelt * 725.0

        # --------------------------------------------------------
        # Time cadence between frames
        # --------------------------------------------------------
        t1_str = extract_datetime_from_header(headers[i - 1])
        t2_str = extract_datetime_from_header(headers[i])

        t1 = Time(t1_str.replace(" UT", ""), format='iso', scale='utc')
        t2 = Time(t2_str.replace(" UT", ""), format='iso', scale='utc')

        F = (t2 - t1).sec  # time difference in seconds

        print(f"Cadence time (F): {F} seconds")

        # Final velocity (km/s)
        magnitude = (velocity * D) / F

        # --------------------------------------------------------
        # Noise filtering (unchanged logic)
        # --------------------------------------------------------
        # Additional pixel displacement threshold (anti-noise)
        pixel_disp = np.sqrt(u**2 + v**2)
        max_pixel_disp = 70
        pixel_disp_mask = pixel_disp < max_pixel_disp

        # Physical velocity threshold
        velocity_mask = (magnitude > lower_velocity) & (magnitude < upper_velocity)

        # Combined mask
        combined_mask = velocity_mask & pixel_disp_mask

        u_filtered = np.where(combined_mask, u, 0)
        v_filtered = np.where(combined_mask, v, 0)
        magnitude_filtered = np.where(combined_mask, magnitude, 0)

        magnitudes.append(magnitude_filtered)
        u_list.append(u_filtered)
        v_list.append(v_filtered)

        prev_frame = next_frame

    return magnitudes, u_list, v_list


# ------------------------------------------------------------
# Heatmap + quiver plot
# ------------------------------------------------------------
def plot_velocity_heatmap_with_quiver(magnitude, u, v, output_dir, headers,
                                      step=15, frame_number=0, date_time_str=""):

    plt.figure(figsize=(10.24, 10.24), dpi=300)

    # Modify turbo colormap so zero appears black
    turbo = plt.cm.turbo(np.linspace(0, 1, 256))
    turbo[0] = [0, 0, 0, 1]
    black_turbo = ListedColormap(turbo)

    plt.imshow(magnitude, cmap=black_turbo, interpolation='none')

    plt.colorbar(label='Velocity Magnitude (km/s)',
                 orientation='horizontal', fraction=0.04, aspect=18, pad=0.1)

    plt.title(f'Velocity Magnitude of CME\n{date_time_str}')

    # --------------------------------------------------------
    # Axis scaling in solar radii
    # --------------------------------------------------------
    cdelt = float(headers[0].get('CDELT1', headers[0].get('cdelt1', 11.9)))
    center_x = magnitude.shape[1] // 2
    center_y = magnitude.shape[0] // 2

    x_extent_arcsec = (np.array([0, magnitude.shape[1] - 1]) - center_x) * cdelt
    y_extent_arcsec = (np.array([0, magnitude.shape[0] - 1]) - center_y) * cdelt

    # Create tick labels that include 0
    num_ticks = 8

    def generate_ticks(min_val, max_val, num, include_zero=True):
        ticks = np.linspace(min_val, max_val, num)
        if include_zero and not np.any(np.isclose(ticks, 0)):
            ticks = np.append(ticks, 0)
            ticks = np.sort(ticks)
        return ticks

    rsun_arcsec = float(headers[0].get('RSUN', headers[0].get('RSUN_ARC', headers[0].get('RSUN_OBS', 959.63))))

    x_rs_ticks = generate_ticks(x_extent_arcsec[0], x_extent_arcsec[1], num_ticks) / rsun_arcsec
    y_rs_ticks = generate_ticks(y_extent_arcsec[0], y_extent_arcsec[1], num_ticks) / rsun_arcsec

    x_pixel_ticks = (x_rs_ticks * rsun_arcsec / cdelt) + center_x
    y_pixel_ticks = (y_rs_ticks * rsun_arcsec / cdelt) + center_y

    plt.xticks(x_pixel_ticks, labels=np.round(x_rs_ticks, 2))
    plt.yticks(y_pixel_ticks, labels=np.round(y_rs_ticks, 2))
    plt.xlabel(r'Solar X ($R_\odot$)')
    plt.ylabel(r'Solar Y ($R_\odot$)')

    # --------------------------------------------------------
    # Subsample arrows for clarity
    # --------------------------------------------------------
    y, x = np.mgrid[0:magnitude.shape[0]:step, 0:magnitude.shape[1]:step]

    u_sub = u[::step, ::step]
    v_sub = v[::step, ::step]

    plt.quiver(x, y, u_sub, -v_sub, color='white',
               scale=500, alpha=0.6, width=0.0025)

    # Save output
    os.makedirs(os.path.join(output_dir, "intensity"), exist_ok=True)

    plt.savefig(os.path.join(output_dir, "intensity",
                             f"frame_{frame_number:04d}.png"),
                dpi=100, bbox_inches='tight', pad_inches=0)

    plt.close()


# ------------------------------------------------------------
# Overlay vectors on original frames
# ------------------------------------------------------------
def plot_velocity_vectors_on_original_frames(frames, magnitudes,
                                             u_list, v_list,
                                             output_dir, step=15,
                                             x_value=0, y_value=0):

    os.makedirs(os.path.join(output_dir, "frames_with_vectors"), exist_ok=True)

    for i, (frame, magnitude, u, v) in enumerate(zip(frames, magnitudes, u_list, v_list)):

        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_GRAY2RGB)

        # Apply same ROI used in OF
        roi = set_ROI(frame_rgb, x_value, y_value,
                      magnitude.shape[1], magnitude.shape[0])

        y, x = np.mgrid[0:magnitude.shape[0]:step, 0:magnitude.shape[1]:step]

        norm = plt.Normalize(vmin=np.min(magnitude), vmax=np.max(magnitude))

        turbo = plt.cm.turbo(np.linspace(0, 1, 256))
        turbo[0] = [0, 0, 0, 1]
        colormap = ListedColormap(turbo)

        arrow_scale = 2  # Parameter preserved from LASCO C2

        for j in range(x.shape[0]):
            for k in range(x.shape[1]):

                y_pos = y[j, k]
                x_pos = x[j, k]

                dx = int(u[y_pos, x_pos] * arrow_scale)
                dy = int(v[y_pos, x_pos] * arrow_scale)

                color = tuple(int(255 * c) for c in colormap(norm(magnitude[y_pos, x_pos]))[:3])

                cv2.arrowedLine(roi,
                                (x_pos, y_pos),
                                (x_pos + dx, y_pos + dy),
                                color, 2, tipLength=0.4)

        plt.figure(figsize=(10.24, 10.24), dpi=200)
        plt.imshow(frame_rgb, interpolation='none')
        plt.axis('off')

        # Add colorbar
        sm = plt.cm.ScalarMappable(norm=norm, cmap=colormap)
        cbar = plt.colorbar(sm, ax=plt.gca(), orientation='horizontal', fraction=0.04, aspect=18, pad=0.00001)
        cbar.set_label('Velocity Magnitude (km/s)', fontsize=10)

        ticks = np.linspace(np.min(magnitude), np.max(magnitude), num=5)
        cbar.set_ticks(ticks)
        cbar.set_ticklabels([f"{tick:.2f}" for tick in ticks])
        cbar.ax.tick_params(labelsize=8)
        plt.subplots_adjust(right=1.2)

        plt.savefig(os.path.join(output_dir, "frames_with_vectors",
                                 f"frame_{i:04d}.png"),
                    bbox_inches='tight', pad_inches=0)

        plt.close()


# ------------------------------------------------------------
# MAIN
# ------------------------------------------------------------
def main():

    frames_folder = "/full/path/to/difference_images"
    output_dir = "/full/path/to/output"

    # --------------------------------------------------------
    # Plotting switches
    # --------------------------------------------------------
    plot_heatmaps = True
    plot_vectors = True

    # Loading preprocessed difference images
    noisy_frames = load_frames_from_folder(frames_folder)

    # Apply denoising to each frame
    denoised_frames = [reduce_noise(frame) for frame in noisy_frames]

    with open(f"{frames_folder}/difference_headers.pkl", "rb") as f:
        headers = pickle.load(f)

    padding = 200
    pad = padding // 2   # = 100 pixels each side

    full_height, full_width = denoised_frames[0].shape

    # Start ROI after padding
    x_value = pad
    y_value = pad

    # Extract only actual data region
    width = full_width - 2 * pad
    height = full_height - 2 * pad

    # --------------------------------------------------------
    magnitudes, u_list, v_list = compute_optical_flow_and_magnitude(
        denoised_frames, headers,
        x_value, y_value, width, height,
        lower_velocity=50, upper_velocity=700
    )

    # --------------------------------------------------------
    # Save velocity data (magnitudes, u, v, times)
    # --------------------------------------------------------
    os.makedirs(os.path.join(output_dir, "velocity_data"), exist_ok=True)

    # Convert list → array for compact storage
    magnitudes_array = np.array(magnitudes, dtype=np.float32)
    u_array = np.array(u_list, dtype=np.float32)
    v_array = np.array(v_list, dtype=np.float32)

    # Extract timestamps (one per velocity frame)
    times = [extract_datetime_from_header(h) for h in headers[:-1]]

    # Save everything in one compressed file
    np.savez_compressed(
        os.path.join(output_dir, "velocity_data", "velocity_data.npz"),
        magnitudes=magnitudes_array,
        u=u_array,
        v=v_array,
        times=times
    )

    # --------------------------------------------------------
    # Plot heatmaps (optional)
    # --------------------------------------------------------
    if plot_heatmaps:
        for i, (magnitude, u, v) in enumerate(zip(magnitudes, u_list, v_list)):

            date_time_str = extract_datetime_from_header(headers[i])

            plot_velocity_heatmap_with_quiver(
                magnitude, u, v, output_dir, headers,
                step=15, frame_number=i, date_time_str=date_time_str
            )

    # --------------------------------------------------------
    # Plot vector overlays (optional)
    # --------------------------------------------------------
    if plot_vectors:
        plot_velocity_vectors_on_original_frames(
            noisy_frames, magnitudes, u_list, v_list,
            output_dir, step=15,
            x_value=x_value, y_value=y_value
        )

    return magnitudes


if __name__ == "__main__":
    magnitudes = main()